## Silver Layer – Data Cleaning and Transformation  

The Silver layer contains cleaned and refined data derived from the Bronze layer.  
In this layer, data quality issues are addressed, schemas are standardized, and datasets are prepared for analytical and business use.

- **Source:** Bronze Delta tables  
- **Storage:** Databricks Delta Lake (Silver tables)  
- **Transformations:**  
  - Data cleaning (handling nulls and duplicates)  
  - Data type standardization  
  - Filtering invalid or incomplete records

In [0]:
from pyspark.sql.functions import col, to_timestamp, trim, upper

In [0]:
%sql show tables IN olist_bronze

database,tableName,isTemporary
olist_bronze,category_translation,false
olist_bronze,customers,false
olist_bronze,geolocation,false
olist_bronze,order_items,false
olist_bronze,orders,false
olist_bronze,payments,false
olist_bronze,products,false
olist_bronze,reviews,false
olist_bronze,sellers,false


In [0]:
%sql
select * from olist_bronze.orders limit 2

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z


In [0]:
%sql CREATE DATABASE IF NOT EXISTS olist_silver

In [0]:
%sql USE olist_silver

In [0]:
orders_brnz_df = spark.table("olist_bronze.orders")
orders_clean_df = orders_brnz_df.dropDuplicates(['order_id']) # drop duplicates if any

# cleanup the datetiemstamp column
orders_clean_df = orders_clean_df \
    .filter(col("order_id").isNotNull()) \
    .filter(col("customer_id").isNotNull()) \
    .withColumn("order_purchase_ts", to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_deliverd_ts",to_timestamp("order_delivered_customer_date")) \
    .select("order_id", "customer_id", "order_status", "order_purchase_ts", "order_deliverd_ts")

orders_clean_df.write.format("delta").mode("overwrite").saveAsTable("olist_silver.orders")


In [0]:
%sql select * from olist_bronze.customers limit 2

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP


In [0]:
cust_brnz_df = spark.table("olist_bronze.customers")
cust_clean_df = cust_brnz_df.dropDuplicates(['customer_id']) # drop duplicates if any

# cleanup data
cust_silver_df = cust_clean_df \
    .filter(col("customer_id").isNotNull()) \
    .withColumn("customer_city",trim(col("customer_city"))) \
    .withColumn("customer_state", upper(trim(col("customer_state")))) \
    .dropDuplicates(["customer_id"]) \
    .select("customer_id","customer_city","customer_state")

cust_silver_df.write.format("delta").mode("overwrite").saveAsTable("olist_silver.customers")


In [0]:
%sql select * from olist_bronze.products limit 2

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20


In [0]:
prd_brnz_df = spark.table("olist_bronze.products")
prd_silver_df = prd_brnz_df \
    .filter(col("product_id").isNotNull()) \
    .dropDuplicates(["product_id"]) \
    .select("product_id","product_category_name","product_photos_qty","product_weight_g","product_length_cm","product_height_cm","product_width_cm")

prd_silver_df.write.format("delta").mode("overwrite").saveAsTable("olist_silver.products")

In [0]:
%sql select * from olist_bronze.order_items limit 2

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93


In [0]:
ord_item_brn_df = spark.table("olist_bronze.order_items")
ord_item_silver_df = ord_item_brn_df \
    .filter(col("order_id").isNotNull()) \
    .filter(col("product_id").isNotNull()) \
    .filter(col("price") > 0) \
    .dropDuplicates(["order_id","product_id"]) \
    .select("order_item_id","order_id","product_id","price","freight_value")

ord_item_silver_df.write.format("delta").mode("overwrite").saveAsTable("olist_silver.order_items")


In [0]:
%sql select * from olist_bronze.payments limit 2

order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39


In [0]:
pay_brnz_df = spark.table("olist_bronze.payments")
pay_silver_df = pay_brnz_df \
    .filter(col("order_id").isNotNull()) \
    .filter(col("payment_type").isNotNull()) \
    .filter(col("payment_value")>0) \
    .select("order_id","payment_type","payment_value")

pay_silver_df.write.format("delta").mode("overwrite").saveAsTable("olist_silver.payments")

In [0]:
%sql show tables in olist_silver

database,tableName,isTemporary
olist_silver,customers,false
olist_silver,order_items,false
olist_silver,orders,false
olist_silver,payments,false
olist_silver,products,false
